# ST 554 Homework 9
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [1]:
#importing required modules
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

## Reading in Data
For this assignment, I chose to use a dataset on Florida real estate properties sold in 2026. I downloaded this dataset from Kaggle, and it is linked [here](https://www.kaggle.com/datasets/kanchana1990/florida-real-estate-sold-dataset-2026). The data file will also be available in my Github, titled `florida_real_estate_sold_properties_utlimate.csv`.

### Creating a Spark Session
The code below creates a spark session for use with reading in our data and building our models.

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 16:50:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Importing Data
The code below reads in our data using `pandas`.

In [3]:
house_data = pd.read_csv("florida_real_estate_sold_properties_ultimate.csv")
house_data.head()

,type,sub_type,listPrice,lastSoldPrice,sqft,stories,beds,baths,baths_full,baths_full_calc,garage,year_built,zip,sanitized_text
0,single_family,NaN,630000.0,605000,2274.0,1.0,2.0,3.0,2.0,2.0,2.0,2007.0,33446.0,"Beautiful 2 Bedroom + Den, 2.5 Bath Home - Mov..."
1,single_family,NaN,289000.0,285000,2170.0,1.0,3.0,2.0,2.0,2.0,2.0,1980.0,33876.0,Welcome to Florida living at its best! This 3-...
2,condos,condo,449000.0,425000,1722.0,NaN,3.0,2.0,2.0,2.0,2.0,2016.0,33913.0,Best Value in Casella and priced to sell... St...
3,single_family,NaN,599000.0,596000,1699.0,1.0,3.0,3.0,3.0,3.0,NaN,1952.0,33009.0,"Beautifully renovated 3-bedroom, 3-bathroom ho..."
4,single_family,NaN,173500.0,165000,640.0,1.0,1.0,1.0,1.0,1.0,NaN,1971.0,32118.0,Experience the ultimate beachfront lifestyle i...


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe.

In [5]:
FL_houses = spark.createDataFrame(house_data)
FL_houses.show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|
|       condos|   condo| 449000.0|       425000|1722.0|    NaN| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|
|single_family|     NaN| 599000.0|       596000|1699.0|    1.0| 3.0|  3.0|       3.0|            3.0|   Na

## Splitting the Data, Metrics, and Models
This section involves:
- splitting the data into a training and test set using spark MLlib
- choosing and describing a metric for judging the fitted models
- fitting three different classes of models and describing them

### Splitting the Data
The code below uses spark MLlib to split the data into a training and test set. This is done by using the `.randomSplit()` method on a `spark` SQL style dataframe.

In [7]:
train, test = FL_houses.randomSplit([0.8,0.2], seed = 44)
print('Number of training observations:', train.count())
print('Number of test observations:', test.count())

Number of training observations: 8673
Number of test observations: 2220


### Choosing a Metric
For this assignment, I will choose the root mean square error (RMSE) for judging the models. RMSE is a very common metric and is easy to interpret. The lower the RMSE, the better the model fit. The mean square error (MSE) is the average of the difference between observed values and values predicted by the model. We want to minimze MSE because we want our model predictions to be close to what is observed! As such, the RMSE is simply the square-root of the MSE.

### Classes of Models
Since I am interested in using `lastSoldPrice` as the response, I will fit three different classes of *regression* models. The following models will be fit:
- A **linear regression model** with the below features
    - feature1
    - feature2
- A **decision tree regression model** with the below features
    - feature1
    - feature2
- A **random forest regression model** with the below features
    - feature1
    - feature2

## Model Fitting